# Pruebas de calidad: ejecutar Silver y Gold, y verificar la cuarentena

Corre los notebooks Silver y Gold contra el Bronze defectuoso (esquemas `pruebas_calidad`, sin tocar producción) y verifica dos cosas:

**1. Cuarentena en Silver.** Por tabla y motivo compara:
- **Esperado:** lo que dejó definido el generador.
- **Log:** lo que registró `hd_cuarentena_resumen` en esta corrida.
- **Silver:** lo que realmente hay en la tabla Silver de prueba.

**2. Propagación a Gold.** Ninguna fila en cuarentena puede llegar a Gold, y las dimensiones deben contener exactamente las filas `VALIDO`.

Además valida el grupo de control (todo `VALIDO`) y que `duplicados_reales` sea 0. Si algo no coincide, el notebook termina con error.

> No se ejecuta `gld_reporte_excel_correo`: envía un correo real.
> Antes de correr, borra las tablas Silver y Gold del esquema de prueba para partir de cero (solo dentro de `pruebas_calidad`).

In [0]:
import uuid
from pyspark.sql import functions as F

ID_PRUEBA = f"PRUEBA-CUARENTENA-{uuid.uuid4()}"
ESQUEMA_PRUEBA = "pruebas_calidad"

PARAMETROS = {
    "bronze_catalog": "brz_dataco",
    "bronze_schema": ESQUEMA_PRUEBA,
    "bronze_table": "raw_supply_chain_defectuosa",
    "silver_catalog": "slv_dataco",
    "silver_schema": ESQUEMA_PRUEBA,
    "gold_catalog": "gld_dataco",
    "gold_schema": ESQUEMA_PRUEBA,
    "metadata_catalog": "metadata_dataco",
    "metadata_schema": ESQUEMA_PRUEBA,
    "id_ejecucion_manual": ID_PRUEBA,
}

NOTEBOOKS_SILVER = [
    "../02_silver_dimensiones/01_negocio/slv_customers",
    "../02_silver_dimensiones/01_negocio/slv_products",
    "../02_silver_dimensiones/01_negocio/slv_locations",
    "../02_silver_hechos/01_negocio/slv_orders",
    "../02_silver_hechos/01_negocio/slv_orders_details",
]

# Las dimensiones primero: los hechos leen las dimensiones.
NOTEBOOKS_GOLD = [
    "../03_gold_dimensiones/01_negocio/gld_dim_customers",
    "../03_gold_dimensiones/01_negocio/gld_dim_products",
    "../03_gold_dimensiones/01_negocio/gld_dim_locations",
    "../03_gold_dimensiones/01_negocio/gld_dim_fecha",
    "../03_gold_dimensiones/01_negocio/gld_dim_shipping_mode",
    "../03_gold_hechos/01_negocio/gld_fact_orders",
    "../03_gold_hechos/01_negocio/gld_fact_order_details",
    "../03_gold_hechos/01_negocio/gld_order_summary",
]

# Partir de cero: solo se borran tablas dentro del esquema de prueba.
for catalogo in ["slv_dataco", "gld_dataco"]:
    esquema = f"{catalogo}.{ESQUEMA_PRUEBA}"
    assert esquema.endswith(".pruebas_calidad"), "Solo se limpian esquemas de prueba"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esquema}")
    for tabla in spark.sql(f"SHOW TABLES IN {esquema}").collect():
        spark.sql(f"DROP TABLE IF EXISTS {esquema}.{tabla['tableName']}")
        print(f"  tabla de prueba eliminada: {esquema}.{tabla['tableName']}")

print(f"id_ejecucion de la prueba: {ID_PRUEBA}")

In [0]:
for ruta in NOTEBOOKS_SILVER + NOTEBOOKS_GOLD:
    print(f"▶ {ruta}")
    dbutils.notebook.run(ruta, 1800, PARAMETROS)
    print("  ✓ terminó")

In [0]:
METADATA = f"metadata_dataco.{ESQUEMA_PRUEBA}"
SILVER = f"slv_dataco.{ESQUEMA_PRUEBA}"
GOLD = f"gld_dataco.{ESQUEMA_PRUEBA}"
BRONZE = f"brz_dataco.{ESQUEMA_PRUEBA}"

# 1) Esperado vs. log vs. Silver real, por tabla y motivo
df_esperado = spark.table(f"{BRONZE}.casos_esperados")

df_log = (
    spark.table(f"{METADATA}.hd_cuarentena_resumen")
    .filter(F.col("id_ejecucion") == ID_PRUEBA)
    .select("tabla_silver", "motivo_cuarentena", F.col("filas").alias("filas_log"))
)

TABLAS_SILVER = ["hd_pedidos", "md_clientes", "md_productos", "md_ubicaciones", "hd_pedido_detalle"]
df_real = None
for tabla in TABLAS_SILVER:
    parcial = (
        spark.table(f"{SILVER}.{tabla}")
        .filter("estado_calidad = 'CUARENTENA'")
        .groupBy("motivo_cuarentena").count()
        .select(F.lit(tabla).alias("tabla_silver"), "motivo_cuarentena", F.col("count").alias("filas_silver"))
    )
    df_real = parcial if df_real is None else df_real.unionByName(parcial)

df_resultado = (
    df_esperado
    .join(df_log, ["tabla_silver", "motivo_cuarentena"], "left")
    .join(df_real, ["tabla_silver", "motivo_cuarentena"], "left")
    .na.fill(0, ["filas_log", "filas_silver"])
    .withColumn("estado", F.when(
        (F.col("filas_esperadas") == F.col("filas_log")) & (F.col("filas_esperadas") == F.col("filas_silver")),
        F.lit("OK")).otherwise(F.lit("FALLA")))
    .orderBy("tabla_silver", "motivo_cuarentena")
)
display(df_resultado)

# Motivos de cuarentena que nadie esperaba
df_inesperado = df_real.join(df_esperado, ["tabla_silver", "motivo_cuarentena"], "left_anti")
print(f"Motivos de cuarentena no esperados: {df_inesperado.count()}")

In [0]:
# 2) Grupo de control: todo VALIDO en pedidos y detalle
casos = {fila["caso_prueba"]: fila for fila in spark.table(f"{BRONZE}.casos_prueba").collect()}
control = casos["CONTROL_SANO"]

def validos_en_rango(tabla, columna, ini, fin):
    filas = (spark.table(f"{SILVER}.{tabla}")
             .filter(F.col(columna).between(ini, fin))
             .groupBy("estado_calidad").count().collect())
    return sum(f["count"] for f in filas if f["estado_calidad"] == "VALIDO")

valido_pedidos = validos_en_rango("hd_pedidos", "order_id", control["order_id_inicio"], control["order_id_fin"])
valido_detalle = validos_en_rango("hd_pedido_detalle", "order_item_id", control["order_item_id_inicio"], control["order_item_id_fin"])
print(f"Control en hd_pedidos: {valido_pedidos} VALIDO de {control['filas']}")
print(f"Control en hd_pedido_detalle: {valido_detalle} VALIDO de {control['filas']}")

# 3) Deduplicación: duplicados_reales debe ser 0 en las 5 tablas
df_dedup = (
    spark.table(f"{METADATA}.hd_deduplicacion_log")
    .filter(F.col("id_ejecucion") == ID_PRUEBA)
    .select("tabla_silver", "filas_entrada", "filas_salida", "filas_agrupadas", "duplicados_reales")
)
display(df_dedup)

In [0]:
# 4) Propagación a Gold: nada en cuarentena puede llegar a Gold
def contar(sql):
    return spark.sql(sql).first()[0]

verificaciones = []   # (nombre, esperado, obtenido)

def agregar(nombre, esperado, sql):
    try:
        obtenido = contar(sql)
    except Exception as error:
        obtenido = f"ERROR: {str(error)[:120]}"
    verificaciones.append((nombre, esperado, obtenido))

# 4.1 Ninguna clave en cuarentena aparece en Gold
agregar("Gold sin pedidos en cuarentena (desc_ft_orders)", 0,
        f"SELECT COUNT(*) FROM {GOLD}.desc_ft_orders WHERE order_id IN "
        f"(SELECT order_id FROM {SILVER}.hd_pedidos WHERE estado_calidad = 'CUARENTENA')")
agregar("Gold sin ítems en cuarentena (desc_ft_order_details)", 0,
        f"SELECT COUNT(*) FROM {GOLD}.desc_ft_order_details WHERE order_item_id IN "
        f"(SELECT order_item_id FROM {SILVER}.hd_pedido_detalle WHERE estado_calidad = 'CUARENTENA')")
agregar("Gold sin clientes en cuarentena (desc_dim_customers)", 0,
        f"SELECT COUNT(*) FROM {GOLD}.desc_dim_customers WHERE customer_id IN "
        f"(SELECT customer_id FROM {SILVER}.md_clientes WHERE estado_calidad = 'CUARENTENA')")
agregar("Gold sin productos en cuarentena (desc_dim_products)", 0,
        f"SELECT COUNT(*) FROM {GOLD}.desc_dim_products WHERE product_card_id IN "
        f"(SELECT product_card_id FROM {SILVER}.md_productos WHERE estado_calidad = 'CUARENTENA')")
agregar("Gold sin ubicaciones incompletas (desc_dim_locations)", 0,
        f"SELECT COUNT(*) FROM {GOLD}.desc_dim_locations "
        f"WHERE market = '' OR order_region = '' OR order_country = ''")

# 4.2 Por caso: los pedidos y los ítems de cada caso defectuoso no están en Gold
for nombre in ["FECHA_PEDIDO_INVALIDA", "CLIENTE_NO_INFORMADO", "RIESGO_RETRASO_INVALIDO",
               "PRIORIDAD_FECHA_Y_CLIENTE", "PRIORIDAD_CLIENTE_Y_RIESGO"]:
    c = casos[nombre]
    agregar(f"Gold sin pedidos del caso {nombre}", 0,
            f"SELECT COUNT(*) FROM {GOLD}.desc_ft_orders "
            f"WHERE order_id BETWEEN {c['order_id_inicio']} AND {c['order_id_fin']}")
c = casos["CANTIDAD_INVALIDA"]
agregar("Gold sin ítems del caso CANTIDAD_INVALIDA", 0,
        f"SELECT COUNT(*) FROM {GOLD}.desc_ft_order_details "
        f"WHERE order_item_id BETWEEN {c['order_item_id_inicio']} AND {c['order_item_id_fin']}")

# 4.3 Las dimensiones contienen exactamente las filas VALIDO de Silver
agregar("desc_dim_customers vigentes = md_clientes VALIDO",
        contar(f"SELECT COUNT(*) FROM {SILVER}.md_clientes WHERE estado_calidad = 'VALIDO'"),
        f"SELECT COUNT(*) FROM {GOLD}.desc_dim_customers WHERE es_vigente")
agregar("desc_dim_products vigentes = md_productos VALIDO",
        contar(f"SELECT COUNT(*) FROM {SILVER}.md_productos WHERE estado_calidad = 'VALIDO'"),
        f"SELECT COUNT(*) FROM {GOLD}.desc_dim_products WHERE es_vigente")
agregar("desc_dim_locations = md_ubicaciones VALIDO",
        contar(f"SELECT COUNT(*) FROM {SILVER}.md_ubicaciones WHERE estado_calidad = 'VALIDO'"),
        f"SELECT COUNT(*) FROM {GOLD}.desc_dim_locations")

# 4.4 El control sano sí llega a Gold (pedidos no cancelados)
agregar("Gold recibe el grupo de control (pedidos > 0)", True,
        f"SELECT COUNT(*) > 0 FROM {GOLD}.desc_ft_orders "
        f"WHERE order_id BETWEEN {control['order_id_inicio']} AND {control['order_id_fin']}")

df_propagacion = spark.createDataFrame(
    [(n, str(e), str(o), "OK" if e == o else "FALLA") for n, e, o in verificaciones],
    "verificacion STRING, esperado STRING, obtenido STRING, estado STRING"
)
display(df_propagacion)

In [0]:
# Veredicto final: cualquier falla detiene el notebook con error.
fallas = []
for fila in df_resultado.filter("estado = 'FALLA'").collect():
    fallas.append(f"Silver {fila['tabla_silver']}.{fila['motivo_cuarentena']}: esperado {fila['filas_esperadas']}, log {fila['filas_log']}, silver {fila['filas_silver']}")

if df_inesperado.count() > 0:
    fallas.append("hay motivos de cuarentena no esperados")
if valido_pedidos != control["filas"]:
    fallas.append(f"control en hd_pedidos: {valido_pedidos} VALIDO de {control['filas']}")
if valido_detalle != control["filas"]:
    fallas.append(f"control en hd_pedido_detalle: {valido_detalle} VALIDO de {control['filas']}")
if df_dedup.count() != 5:
    fallas.append(f"hd_deduplicacion_log tiene {df_dedup.count()} filas para esta corrida (esperadas 5)")
if df_dedup.filter("duplicados_reales > 0").count() > 0:
    fallas.append("duplicados_reales mayor que 0")
for fila in df_propagacion.filter("estado = 'FALLA'").collect():
    fallas.append(f"Gold: {fila['verificacion']} (esperado {fila['esperado']}, obtenido {fila['obtenido']})")

if fallas:
    raise AssertionError("Prueba de cuarentena con fallas:\n- " + "\n- ".join(fallas))

print("✓ PRUEBA DE CUARENTENA APROBADA: 9 motivos en Silver, prioridad, control sano, 0 duplicados reales y ninguna fila en cuarentena llega a Gold.")